# 05 - Feature Preprocessing

## Objective

This notebook prepares the engineered customer-level features for clustering.

The preprocessing stage will:

- load the registered customer feature dataset;
- select the provisional modelling features;
- handle structurally missing values;
- review feature distributions and skewness;
- assess extreme values and outliers;
- apply suitable transformations;
- scale the final modelling features;
- produce the final clustering feature matrix.

The full engineered customer dataset remains unchanged for traceability and
cluster interpretation.

In [1]:
# necessary libraries
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import Data
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import AccountKeyConfiguration
import mltable
from mltable import MLTableHeaders, MLTableFileEncoding
import pandas as pd
import numpy as np

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

# Load the registered Azure ML Data Asset so that analysis is based on the
# governed MLTable rather than a local file path.
paid_purchase_asset = ml_client.data.get(
    name="online-retail-paid-purchases",
    version="1"
)

# Load the MLTable definition and materialise the dataset as a Pandas DataFrame
# for interactive profiling and exploratory analysis.
retail_table = mltable.load(paid_purchase_asset.path)
df_paid = retail_table.to_pandas_dataframe()

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [2]:
# Retrieve the registered customer-level feature dataset from Azure ML.

customer_feature_asset = ml_client.data.get(
    name="online-retail-customer-features",
    version="1"
)

print(customer_feature_asset.name)
print(customer_feature_asset.version)

online-retail-customer-features
1


In [3]:
# Load the registered MLTable and materialise it as a Pandas DataFrame.

customer_feature_table = mltable.load(
    customer_feature_asset.path
)

df_customers = customer_feature_table.to_pandas_dataframe()

print("Rows:", f"{df_customers.shape[0]:,}")
print("Columns:", df_customers.shape[1])

df_customers.head()

Rows: 4,334
Columns: 23


,CustomerID,LastPurchaseDate,Recency,Frequency,MonetaryValue,AverageOrderValue,UniqueProducts,FirstPurchaseDate,CustomerTenureDays,TotalQuantity,...,PostageSpend,PostageInvoices,PaidInvoices,PostageInvoiceShare,HasPostage,ReturnInvoices,ReturnValue,MerchandiseReturnValue,ObservedMerchandiseReturnValueRate,HasMerchandiseReturn
0,12346,2011-01-18 10:01:00,326,1,77183.60,77183.600000,1,2011-01-18 10:01:00,0,74215,...,0.0,0,1,0.0,0,1,77183.6,77183.6,1.0,1
1,12347,2011-12-07 15:52:00,2,7,4310.00,615.714286,103,2010-12-07 14:57:00,365,2458,...,0.0,0,7,0.0,0,0,0.0,0.0,0.0,0
2,12348,2011-09-25 13:13:00,75,4,1437.24,359.310000,21,2010-12-16 19:09:00,282,2332,...,360.0,4,4,1.0,1,0,0.0,0.0,0.0,0
3,12349,2011-11-21 09:51:00,19,1,1457.55,1457.550000,72,2011-11-21 09:51:00,0,630,...,300.0,1,1,1.0,1,0,0.0,0.0,0.0,0
4,12350,2011-02-02 16:01:00,310,1,294.40,294.400000,16,2011-02-02 16:01:00,0,196,...,40.0,1,1,1.0,1,0,0.0,0.0,0.0,0


In [4]:
df_customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4334 entries, 0 to 4333
Data columns (total 23 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   CustomerID                          4334 non-null   Int64         
 1   LastPurchaseDate                    4334 non-null   datetime64[ns]
 2   Recency                             4334 non-null   int64         
 3   Frequency                           4334 non-null   int64         
 4   MonetaryValue                       4334 non-null   float64       
 5   AverageOrderValue                   4334 non-null   float64       
 6   UniqueProducts                      4334 non-null   int64         
 7   FirstPurchaseDate                   4334 non-null   datetime64[ns]
 8   CustomerTenureDays                  4334 non-null   int64         
 9   TotalQuantity                       4334 non-null   int64         
 10  AverageQuantityPerOrder 

## 1. Initial Modelling Feature Selection

The feature-engineering stage produced 15 provisional clustering variables.

`AverageDaysBetweenPurchases` contains structurally missing values for
single-purchase customers. It is also derived from customer tenure and purchase
frequency, meaning it largely duplicates information already represented by
`CustomerTenureDays` and `Frequency`.

Rather than introducing artificial imputation values, the feature is retained
in the full customer dataset for interpretation but excluded from the modelling
feature set.

This leaves 14 behavioural features for further preprocessing.

In [5]:
# Define the behavioural features that will proceed to distribution,
# outlier and transformation assessment.

model_feature_columns = [
    "Recency",
    "Frequency",
    "MonetaryValue",
    "AverageOrderValue",
    "UniqueProducts",
    "CustomerTenureDays",
    "TotalQuantity",
    "AverageQuantityPerOrder",
    "PostageSpend",
    "PostageInvoices",
    "PostageInvoiceShare",
    "ReturnInvoices",
    "MerchandiseReturnValue",
    "ObservedMerchandiseReturnValueRate"
]

print("Model features:", len(model_feature_columns))

Model features: 14


In [6]:
# Create a separate modelling dataset while leaving the full
# customer-level dataset unchanged for traceability.

X = df_customers[
    model_feature_columns
].copy()

print("Modelling dataset shape:", X.shape)

Modelling dataset shape: (4334, 14)


In [7]:
# Confirm that the selected modelling features contain no missing values.

X.isna().sum()

Recency                               0
Frequency                             0
MonetaryValue                         0
AverageOrderValue                     0
UniqueProducts                        0
CustomerTenureDays                    0
TotalQuantity                         0
AverageQuantityPerOrder               0
PostageSpend                          0
PostageInvoices                       0
PostageInvoiceShare                   0
ReturnInvoices                        0
MerchandiseReturnValue                0
ObservedMerchandiseReturnValueRate    0
dtype: int64

## 2. Feature Skewness Assessment

Clustering algorithms such as K-Means are sensitive to the distribution and
scale of numeric features.

Several customer behavioural measures, including spend, purchase frequency and
quantity, may contain strongly right-skewed distributions caused by a relatively
small number of highly active or high-value customers.

Skewness is therefore assessed before deciding which features require
transformation. No transformation is applied at this stage.

In [8]:
# Measure the skewness of each candidate modelling feature.
# Positive values indicate longer right tails, while negative values
# indicate longer left tails.

feature_skewness = (
    X
    .skew()
    .sort_values(ascending=False)
    .to_frame(name="Skewness")
)

feature_skewness["AbsoluteSkewness"] = (
    feature_skewness["Skewness"].abs()
)

feature_skewness

,Skewness,AbsoluteSkewness
MerchandiseReturnValue,52.024506,52.024506
AverageQuantityPerOrder,47.667920,47.667920
AverageOrderValue,41.457561,41.457561
PostageSpend,38.446690,38.446690
TotalQuantity,20.413210,20.413210
MonetaryValue,19.551828,19.551828
Frequency,11.976299,11.976299
ObservedMerchandiseReturnValueRate,10.751621,10.751621
PostageInvoices,10.340991,10.340991
ReturnInvoices,8.826384,8.826384


Skewness close to 0   → fairly symmetrical

±0.5 to ±1.0          → moderately skewed

greater than ±1.0     → strongly skewed

### Zero-value assessment

Several behavioural features, particularly postage and return measures, may be
highly skewed because a large proportion of customers have no observed activity
for that behaviour.

Before applying transformations, the proportion of zero values is assessed.
This helps distinguish conventional right-skewness from zero-inflated features,
where most customers have a value of zero and a smaller group exhibit positive
activity.

In [9]:
# Measure the number and proportion of zero values in each modelling feature.

zero_summary = pd.DataFrame({
    "ZeroCount": (X == 0).sum(),
    "ZeroPercent": ((X == 0).sum() / len(X) * 100).round(2)
})

zero_summary = (
    zero_summary
    .sort_values("ZeroPercent", ascending=False)
)

zero_summary

,ZeroCount,ZeroPercent
PostageSpend,4002,92.34
PostageInvoices,4002,92.34
PostageInvoiceShare,4002,92.34
MerchandiseReturnValue,2828,65.25
ObservedMerchandiseReturnValueRate,2828,65.25
ReturnInvoices,2781,64.17
CustomerTenureDays,1564,36.09
UniqueProducts,0,0.00
MonetaryValue,0,0.00
Frequency,0,0.00


### Positive-value skewness for sparse behavioural features

Postage and return measures contain a substantial proportion of zero values,
representing customers with no observed activity.

To distinguish zero inflation from skewness among active customers, the
distribution of positive values is assessed separately before determining
the appropriate preprocessing strategy.

In [10]:
# Assess skewness among customers with positive activity only.
# This separates the effect of many zero observations from the
# distribution of the behaviour among customers who actually exhibit it.

sparse_features = [
    "PostageSpend",
    "PostageInvoices",
    "PostageInvoiceShare",
    "ReturnInvoices",
    "MerchandiseReturnValue",
    "ObservedMerchandiseReturnValueRate"
]

positive_value_summary = []

for feature in sparse_features:
    positive_values = X.loc[X[feature] > 0, feature]

    positive_value_summary.append({
        "Feature": feature,
        "PositiveCustomers": len(positive_values),
        "PositivePercent": round(
            len(positive_values) / len(X) * 100,
            2
        ),
        "PositiveSkewness": round(
            positive_values.skew(),
            4
        )
    })

positive_value_summary = pd.DataFrame(
    positive_value_summary
).sort_values(
    "PositiveSkewness",
    ascending=False
)

positive_value_summary

,Feature,PositiveCustomers,PositivePercent,PositiveSkewness
4,MerchandiseReturnValue,1506,34.75,30.6854
0,PostageSpend,332,7.66,11.3846
3,ReturnInvoices,1553,35.83,6.9379
5,ObservedMerchandiseReturnValueRate,1506,34.75,6.6538
1,PostageInvoices,332,7.66,3.3019
2,PostageInvoiceShare,332,7.66,-1.5164


## 3. Transformation Strategy

The skewness assessment shows that several customer-behaviour features have
strong right-skewed distributions.

For positively valued monetary, quantity and count measures, a `log1p`
transformation will be used. `log1p(x)` calculates `log(1 + x)`, allowing
zero-valued observations to remain valid while compressing extreme values.

The following features will be log-transformed:

- Recency
- Frequency
- MonetaryValue
- AverageOrderValue
- UniqueProducts
- TotalQuantity
- AverageQuantityPerOrder
- PostageSpend
- PostageInvoices
- ReturnInvoices
- MerchandiseReturnValue
- ObservedMerchandiseReturnValueRate

`CustomerTenureDays` is retained without transformation because its distribution
is comparatively balanced.

`PostageInvoiceShare` is also retained in its original form because it is a
bounded proportion between 0 and 1 and its positive-value distribution does not
show the same right-skew pattern as the magnitude-based features.

In [11]:
# Features requiring log transformation because of substantial
# positive skewness and/or extreme upper-tail values.

log_transform_features = [
    "Recency",
    "Frequency",
    "MonetaryValue",
    "AverageOrderValue",
    "UniqueProducts",
    "TotalQuantity",
    "AverageQuantityPerOrder",
    "PostageSpend",
    "PostageInvoices",
    "ReturnInvoices",
    "MerchandiseReturnValue",
    "ObservedMerchandiseReturnValueRate"
]

# Features retained on their original scale before standardisation.

untransformed_features = [
    "CustomerTenureDays",
    "PostageInvoiceShare"
]

print("Log-transformed features:", len(log_transform_features))
print("Untransformed features:", len(untransformed_features))

Log-transformed features: 12
Untransformed features: 2


In [12]:
# Create a separate copy so the original modelling features remain
# available for comparison and interpretation.

X_transformed = X.copy()

for feature in log_transform_features:
    X_transformed[feature] = np.log1p(
        X_transformed[feature]
    )

In [13]:
# Confirm that transformation produced no missing or infinite values.

print(
    "Missing values:",
    X_transformed.isna().sum().sum()
)

print(
    "Infinite values:",
    np.isinf(
        X_transformed.select_dtypes(include=np.number)
    ).sum().sum()
)

Missing values: 0
Infinite values: 0


## 4. Post-Transformation Skewness Review

The selected features have been transformed using `log1p` to reduce the
influence of extreme right tails while preserving zero-valued observations.

Skewness is recalculated after transformation to assess whether the feature
distributions have become more suitable for distance-based clustering.

In [14]:
# Recalculate skewness after transformation.

transformed_skewness = (
    X_transformed
    .skew()
    .sort_values(ascending=False)
    .to_frame(name="SkewnessAfter")
)

transformed_skewness

,SkewnessAfter
ObservedMerchandiseReturnValueRate,7.637962
PostageInvoices,4.594527
PostageInvoiceShare,3.569897
PostageSpend,3.564193
ReturnInvoices,1.650845
MerchandiseReturnValue,1.307753
Frequency,1.214115
CustomerTenureDays,0.456490
MonetaryValue,0.400774
AverageOrderValue,0.283344


In [15]:
# Compare feature skewness before and after transformation.

skewness_comparison = pd.DataFrame({
    "SkewnessBefore": X.skew(),
    "SkewnessAfter": X_transformed.skew()
})

skewness_comparison["AbsoluteBefore"] = (
    skewness_comparison["SkewnessBefore"].abs()
)

skewness_comparison["AbsoluteAfter"] = (
    skewness_comparison["SkewnessAfter"].abs()
)

skewness_comparison["Improvement"] = (
    skewness_comparison["AbsoluteBefore"]
    - skewness_comparison["AbsoluteAfter"]
)

skewness_comparison = (
    skewness_comparison
    .sort_values("AbsoluteAfter", ascending=False)
)

skewness_comparison.round(4)

,SkewnessBefore,SkewnessAfter,AbsoluteBefore,AbsoluteAfter,Improvement
ObservedMerchandiseReturnValueRate,10.7516,7.6380,10.7516,7.6380,3.1137
PostageInvoices,10.3410,4.5945,10.3410,4.5945,5.7465
PostageInvoiceShare,3.5699,3.5699,3.5699,3.5699,0.0000
PostageSpend,38.4467,3.5642,38.4467,3.5642,34.8825
ReturnInvoices,8.8264,1.6508,8.8264,1.6508,7.1755
MerchandiseReturnValue,52.0245,1.3078,52.0245,1.3078,50.7168
Frequency,11.9763,1.2141,11.9763,1.2141,10.7622
CustomerTenureDays,0.4565,0.4565,0.4565,0.4565,0.0000
MonetaryValue,19.5518,0.4008,19.5518,0.4008,19.1511
Recency,1.2429,-0.3789,1.2429,0.3789,0.8640


### Post-transformation assessment

The `log1p` transformation substantially reduced skewness for the core
continuous behavioural features, including monetary value, quantity, product
diversity and average order measures.

Several postage and return features remain strongly skewed after transformation.
This is expected because these variables are zero-inflated: most customers have
no observed postage or return activity, while a smaller group has positive
values.

These zero values represent genuine customer behaviour rather than data quality
issues. No additional transformation is applied solely to force these
distributions towards normality.

The next preprocessing step therefore focuses on extreme-value assessment and
the potential influence of outliers on distance-based clustering.

In [16]:
# Assess potential extreme values in the transformed feature space
# using the interquartile range (IQR) method.

outlier_summary = []

for feature in X_transformed.columns:
    q1 = X_transformed[feature].quantile(0.25)
    q3 = X_transformed[feature].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    outlier_count = (
        (X_transformed[feature] < lower_bound)
        | (X_transformed[feature] > upper_bound)
    ).sum()

    outlier_summary.append({
        "Feature": feature,
        "LowerBound": lower_bound,
        "UpperBound": upper_bound,
        "OutlierCount": outlier_count,
        "OutlierPercent": (
            outlier_count / len(X_transformed) * 100
        )
    })

outlier_summary = pd.DataFrame(outlier_summary)

outlier_summary = (
    outlier_summary
    .sort_values("OutlierPercent", ascending=False)
    .reset_index(drop=True)
)

outlier_summary.round(2)

,Feature,LowerBound,UpperBound,OutlierCount,OutlierPercent
0,ObservedMerchandiseReturnValueRate,-0.01,0.02,658,15.18
1,PostageInvoices,0.00,0.00,332,7.66
2,PostageSpend,0.00,0.00,332,7.66
3,PostageInvoiceShare,0.00,0.00,332,7.66
4,ReturnInvoices,-1.04,1.73,150,3.46
5,AverageOrderValue,3.88,7.35,130,3.00
6,AverageQuantityPerOrder,2.93,7.22,120,2.77
7,TotalQuantity,2.34,9.63,63,1.45
8,MonetaryValue,3.21,9.91,52,1.20
9,Frequency,-0.95,3.44,42,0.97


### Outlier assessment outcome

The IQR assessment identifies apparent outliers in several features, but the
results require different interpretation depending on the feature type.

For zero-inflated postage and return measures, the IQR method is not an
appropriate basis for record removal because the large concentration of zero
values causes genuine positive activity to be classified as anomalous.

Following log transformation, the core monetary, quantity and frequency
features contain relatively small proportions of IQR-defined extreme values.

No customers are removed automatically. Extreme observations will instead be
reviewed to determine whether they represent plausible high-value or high-volume
customer behaviour before any capping decision is made.

In [17]:
# Review upper-tail behaviour for the core continuous features after
# transformation, excluding sparse postage and return measures.

core_outlier_features = [
    "Frequency",
    "MonetaryValue",
    "AverageOrderValue",
    "UniqueProducts",
    "TotalQuantity",
    "AverageQuantityPerOrder"
]

upper_tail_summary = (
    X_transformed[core_outlier_features]
    .quantile([0.95, 0.99, 0.995, 1.00])
    .T
)

upper_tail_summary.columns = [
    "P95",
    "P99",
    "P99_5",
    "Maximum"
]

upper_tail_summary["Max_to_P99"] = (
    upper_tail_summary["Maximum"]
    / upper_tail_summary["P99"]
)

upper_tail_summary.round(3)

,P95,P99,P99_5,Maximum,Max_to_P99
Frequency,2.639,3.434,3.761,5.333,1.553
MonetaryValue,8.655,9.837,10.622,12.539,1.275
AverageOrderValue,6.820,7.613,8.105,11.341,1.490
UniqueProducts,5.323,5.871,6.090,7.488,1.275
TotalQuantity,8.177,9.296,10.182,12.190,1.311
AverageQuantityPerOrder,6.399,7.234,7.539,11.215,1.550


### Extreme-value treatment decision

Following `log1p` transformation, the upper tails of the core behavioural
features are substantially compressed.

The maximum transformed values are approximately 1.3 to 1.6 times their
respective 99th percentiles. This does not indicate sufficiently extreme
separation to justify automatic capping or customer removal.

High-value, high-frequency and high-volume customers may represent legitimate
and commercially important customer segments. Their behaviour is therefore
preserved.

No winsorisation, clipping or outlier-based record removal is applied at this
stage.

## 5. Post-Transformation Feature Redundancy Review

Feature relationships are reassessed after transformation because extreme
skewness can inflate Pearson correlations in the original feature space.

This review helps determine whether strongly overlapping variables would give
particular customer behaviours disproportionate influence during clustering.

In [18]:
# Recalculate feature correlations after transformation.

transformed_correlation = X_transformed.corr(
    method="pearson"
)

In [19]:
# Identify unique transformed feature pairs with an absolute
# correlation of at least 0.80.

strong_transformed_correlations = (
    transformed_correlation
    .where(
        np.triu(
            np.ones(transformed_correlation.shape),
            k=1
        ).astype(bool)
    )
    .stack()
    .reset_index()
)

strong_transformed_correlations.columns = [
    "Feature1",
    "Feature2",
    "Correlation"
]

strong_transformed_correlations["AbsoluteCorrelation"] = (
    strong_transformed_correlations["Correlation"].abs()
)

strong_transformed_correlations = (
    strong_transformed_correlations.loc[
        strong_transformed_correlations["AbsoluteCorrelation"] >= 0.80
    ]
    .sort_values(
        "AbsoluteCorrelation",
        ascending=False
    )
    .reset_index(drop=True)
)

strong_transformed_correlations.round(4)

,Feature1,Feature2,Correlation,AbsoluteCorrelation
0,PostageSpend,PostageInvoices,0.9421,0.9421
1,MonetaryValue,TotalQuantity,0.9272,0.9272
2,PostageSpend,PostageInvoiceShare,0.9245,0.9245
3,ReturnInvoices,MerchandiseReturnValue,0.8837,0.8837
4,PostageInvoices,PostageInvoiceShare,0.8653,0.8653
5,AverageOrderValue,AverageQuantityPerOrder,0.8242,0.8242
6,Frequency,CustomerTenureDays,0.8108,0.8108
7,Frequency,MonetaryValue,0.8081,0.8081


## 6. Final Clustering Feature Selection

Following transformation and redundancy assessment, a reduced set of
behaviourally distinct features is selected for clustering.

Highly overlapping measures are excluded from the modelling matrix to avoid
giving particular behaviours disproportionate influence in distance-based
clustering.

The final feature set represents customer recency, purchase frequency,
economic value, product breadth, relationship tenure, typical order volume,
postage behaviour and return behaviour.

Features excluded from clustering remain available in the full customer
dataset for subsequent cluster interpretation and profiling.

In [20]:
# Define the final behavioural features to be supplied to the
# clustering preprocessing pipeline.

final_model_features = [
    "Recency",
    "Frequency",
    "MonetaryValue",
    "UniqueProducts",
    "CustomerTenureDays",
    "AverageQuantityPerOrder",
    "PostageInvoiceShare",
    "ObservedMerchandiseReturnValueRate"
]

print("Final clustering features:", len(final_model_features))

Final clustering features: 8


In [21]:
# Create the final transformed modelling dataset.
# Features previously selected for log transformation use their transformed
# values, while tenure and postage share remain on their original scale.

X_final = X_transformed[
    final_model_features
].copy()

print("Final modelling shape:", X_final.shape)

X_final.head()

Final modelling shape: (4334, 8)


,Recency,Frequency,MonetaryValue,UniqueProducts,CustomerTenureDays,AverageQuantityPerOrder,PostageInvoiceShare,ObservedMerchandiseReturnValueRate
0,5.789960,0.693147,11.253955,0.693147,0,11.214735,0.0,0.693147
1,1.098612,2.079442,8.368925,4.644391,365,5.864037,0.0,0.000000
2,4.330733,1.609438,7.271175,3.091042,282,6.369901,1.0,0.000000
3,2.995732,0.693147,7.285198,4.290459,0,6.447306,1.0,0.000000
4,5.739793,0.693147,5.688330,2.833213,0,5.283204,1.0,0.000000


## 7. Feature Scaling

K-Means clustering is based on distances between observations. Features with
larger numerical ranges can therefore have disproportionate influence on the
result.

The final clustering features are standardised using `StandardScaler`, which
centres each feature around a mean of 0 and scales it to a standard deviation
of approximately 1.

Scaling is applied after feature transformation and final feature selection.

In [22]:
from sklearn.preprocessing import StandardScaler

# Fit StandardScaler on the final transformed behavioural features
# and convert all features to a comparable numerical scale.

scaler = StandardScaler()

X_scaled_array = scaler.fit_transform(X_final)

In [23]:
# Convert the scaled NumPy array back to a DataFrame while preserving
# the original customer row index and feature names.

X_scaled = pd.DataFrame(
    X_scaled_array,
    columns=final_model_features,
    index=X_final.index
)

X_scaled.head()

,Recency,Frequency,MonetaryValue,UniqueProducts,CustomerTenureDays,AverageQuantityPerOrder,PostageInvoiceShare,ObservedMerchandiseReturnValueRate
0,1.460438,-0.951243,3.729311,-2.523812,-0.986461,6.873723,-0.270652,10.364965
1,-2.038956,1.082163,1.429190,0.966888,1.777127,0.908605,-0.270652,-0.282114
2,0.371964,0.392765,0.553997,-0.405407,1.148694,1.472557,4.005881,-0.282114
3,-0.623847,-0.951243,0.565177,0.654210,-0.986461,1.558851,4.005881,-0.282114
4,1.423017,-0.951243,-0.707943,-0.633185,-0.986461,0.261075,4.005881,-0.282114


In [24]:
print("Scaled modelling shape:", X_scaled.shape)

print(
    "Missing values:",
    X_scaled.isna().sum().sum()
)

print(
    "Infinite values:",
    np.isinf(X_scaled).sum().sum()
)

Scaled modelling shape: (4334, 8)
Missing values: 0
Infinite values: 0


In [25]:
# Confirm that scaling has centred the features close to zero
# with standard deviations close to one.

scaling_check = pd.DataFrame({
    "Mean": X_scaled.mean(),
    "StandardDeviation": X_scaled.std(ddof=0)
})

scaling_check.round(4)

,Mean,StandardDeviation
Recency,0.0,1.0
Frequency,-0.0,1.0
MonetaryValue,-0.0,1.0
UniqueProducts,-0.0,1.0
CustomerTenureDays,-0.0,1.0
AverageQuantityPerOrder,-0.0,1.0
PostageInvoiceShare,-0.0,1.0
ObservedMerchandiseReturnValueRate,-0.0,1.0


## 8. Persist Preprocessed Clustering Dataset

The final scaled feature matrix is combined with `CustomerID` for traceability.

`CustomerID` is retained only as an identifier and will not be supplied to the
clustering algorithm.

The resulting dataset represents the final preprocessed input for model
development.

In [26]:
# Combine the customer identifier with the final scaled feature matrix.
# CustomerID is retained for traceability but is not a clustering feature.

df_clustering_input = pd.concat(
    [
        df_customers.loc[X_scaled.index, ["CustomerID"]],
        X_scaled
    ],
    axis=1
)

print("Clustering input shape:", df_clustering_input.shape)

df_clustering_input.head()

Clustering input shape: (4334, 9)


,CustomerID,Recency,Frequency,MonetaryValue,UniqueProducts,CustomerTenureDays,AverageQuantityPerOrder,PostageInvoiceShare,ObservedMerchandiseReturnValueRate
0,12346,1.460438,-0.951243,3.729311,-2.523812,-0.986461,6.873723,-0.270652,10.364965
1,12347,-2.038956,1.082163,1.429190,0.966888,1.777127,0.908605,-0.270652,-0.282114
2,12348,0.371964,0.392765,0.553997,-0.405407,1.148694,1.472557,4.005881,-0.282114
3,12349,-0.623847,-0.951243,0.565177,0.654210,-0.986461,1.558851,4.005881,-0.282114
4,12350,1.423017,-0.951243,-0.707943,-0.633185,-0.986461,0.261075,4.005881,-0.282114


In [27]:
# Confirm that each row still represents exactly one customer.

assert len(df_clustering_input) == df_clustering_input["CustomerID"].nunique()

print("Customer uniqueness validation passed.")

Customer uniqueness validation passed.


In [28]:
# Persist the final scaled clustering dataset in Parquet format.

clustering_input_path = (
    "../data/processed/online_retail_clustering_input.parquet"
)

df_clustering_input.to_parquet(
    clustering_input_path,
    engine="pyarrow",
    index=False
)

print(
    f"Clustering input saved to: "
    f"{clustering_input_path}"
)

Clustering input saved to: ../data/processed/online_retail_clustering_input.parquet


In [29]:
# Reload the persisted dataset to confirm successful serialisation.

clustering_input_check = pd.read_parquet(
    clustering_input_path
)

print("Original shape:", df_clustering_input.shape)
print("Saved shape:", clustering_input_check.shape)

assert (
    clustering_input_check.shape
    == df_clustering_input.shape
)

print("Clustering input validation passed.")

Original shape: (4334, 9)
Saved shape: (4334, 9)
Clustering input validation passed.


## 9. Persist Preprocessing Artefacts

The fitted `StandardScaler` and final feature order are persisted so that the
same preprocessing logic can be reproduced during model training and future
customer scoring.

The scaler must not be re-fitted on new scoring data, as this would change the
feature representation used by the clustering model.

In [30]:
import joblib
import json
import os

# Create a dedicated project folder for reusable preprocessing artefacts.

preprocessing_artifact_folder = "../deployment/preprocessing"

os.makedirs(
    preprocessing_artifact_folder,
    exist_ok=True
)

In [31]:
# Persist the fitted StandardScaler.
# This preserves the means and standard deviations learned from
# the current customer population.

scaler_path = os.path.join(
    preprocessing_artifact_folder,
    "standard_scaler.joblib"
)

joblib.dump(
    scaler,
    scaler_path
)

print(f"Scaler saved to: {scaler_path}")

Scaler saved to: ../deployment/preprocessing/standard_scaler.joblib


In [32]:
# Persist the exact modelling feature order expected by the scaler
# and downstream clustering model.

feature_order_path = os.path.join(
    preprocessing_artifact_folder,
    "feature_order.json"
)

with open(feature_order_path, "w") as file:
    json.dump(
        final_model_features,
        file,
        indent=4
    )

print(f"Feature order saved to: {feature_order_path}")

Feature order saved to: ../deployment/preprocessing/feature_order.json


In [33]:
os.listdir(preprocessing_artifact_folder)

['feature_order.json', 'standard_scaler.joblib']

In [34]:
# Reload the persisted preprocessing artefacts to confirm that they
# can be used independently of the current notebook session.

loaded_scaler = joblib.load(
    scaler_path
)

with open(feature_order_path, "r") as file:
    loaded_feature_order = json.load(file)

print("Loaded features:", loaded_feature_order)
print("Number of features:", len(loaded_feature_order))

Loaded features: ['Recency', 'Frequency', 'MonetaryValue', 'UniqueProducts', 'CustomerTenureDays', 'AverageQuantityPerOrder', 'PostageInvoiceShare', 'ObservedMerchandiseReturnValueRate']
Number of features: 8


In [35]:
# Register the final preprocessed clustering dataset as a versioned
# Azure ML URI_FILE data asset.

clustering_input_parquet_asset = Data(
    name="online-retail-clustering-input-parquet",
    version="1",
    type=AssetTypes.URI_FILE,
    path=clustering_input_path,
    description=(
        "Final scaled customer feature dataset used as input "
        "for customer segmentation clustering."
    )
)

registered_clustering_input_parquet = ml_client.data.create_or_update(
    clustering_input_parquet_asset
)

In [37]:
print("Name:", registered_clustering_input_parquet.name)
print("Version:", registered_clustering_input_parquet.version)
print("Type:", registered_clustering_input_parquet.type)
print("Path:", registered_clustering_input_parquet.path)

Name: online-retail-clustering-input-parquet
Version: 1
Type: uri_file
Path: azureml://subscriptions/9d0e4acf-2675-4581-90be-c8f45f73d333/resourcegroups/clustering-project/workspaces/clustering-workspace/datastores/workspaceblobstore/paths/LocalUpload/4abc9e7610e799a1acc5695df79cabb3a7e533079ac0cfcccb8aa12a063bd8fb/online_retail_clustering_input.parquet


In [38]:
# Create an MLTable definition that points to the registered
# final clustering-input Parquet dataset in Azure storage.

clustering_input_table = mltable.from_parquet_files(
    paths=[
        {
            "file": registered_clustering_input_parquet.path
        }
    ]
)

In [39]:
# Materialise the MLTable to confirm that the final preprocessed
# clustering dataset can be loaded successfully.

clustering_input_test = (
    clustering_input_table
    .to_pandas_dataframe()
)

print(
    "Clustering input MLTable shape:",
    clustering_input_test.shape
)

Clustering input MLTable shape: (4334, 9)


In [40]:
# Create a project folder for the final clustering-input MLTable definition.

clustering_input_mltable_folder = "../data/mltable/clustering-input"

os.makedirs(
    clustering_input_mltable_folder,
    exist_ok=True
)

In [41]:
# Save the MLTable definition file.

clustering_input_table.save(
    clustering_input_mltable_folder
)

paths:
- file: azureml://subscriptions/9d0e4acf-2675-4581-90be-c8f45f73d333/resourcegroups/clustering-project/workspaces/clustering-workspace/datastores/workspaceblobstore/paths/LocalUpload/4abc9e7610e799a1acc5695df79cabb3a7e533079ac0cfcccb8aa12a063bd8fb/online_retail_clustering_input.parquet
transformations:
- read_parquet:
    include_path_column: false
    path_column: Path
type: mltable

In [42]:
# Register the final clustering-input MLTable as a versioned Azure ML data asset.

clustering_input_mltable_asset = Data(
    name="online-retail-clustering-input",
    version="1",
    type=AssetTypes.MLTABLE,
    path=clustering_input_mltable_folder,
    description=(
        "Final preprocessed and scaled customer feature dataset "
        "used for clustering model development."
    )
)

registered_clustering_input_mltable = ml_client.data.create_or_update(
    clustering_input_mltable_asset
)

Uploading clustering-input (0.0 MBs): 100%|██████████| 394/394 [00:00<00:00, 18805.97it/s]




In [43]:
print("Name:", registered_clustering_input_mltable.name)
print("Version:", registered_clustering_input_mltable.version)
print("Type:", registered_clustering_input_mltable.type)

Name: online-retail-clustering-input
Version: 1
Type: mltable
